Lets record the audio from the esp32!

In [6]:
# Run once in this notebook kernel.
%pip install sounddevice ipywidgets requests


   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 23.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [18]:
from pathlib import Path
from datetime import datetime
import struct
import requests
import wave
from IPython.display import Audio, display

receiver = "192.168.1.143"
seconds = 10
monitor_volume = 1.0  # 0 = mute, 1 = original, 2 = double; affects speakers only
output_device = None  # Default computer speakers; see sounddevice.query_devices()

def record_receiver(receiver, seconds):
    try:
        import sounddevice as sd
    except ImportError:
        print("Run the setup cell above (%pip install sounddevice), then rerun this cell.")
        return
    if not isinstance(seconds, int) or not 1 <= seconds <= 60:
        raise ValueError("Choose an integer duration from 1 to 60 seconds.")
    if not 0 <= monitor_volume <= 20:
        raise ValueError("monitor_volume must be between 0 and 20.")
    print("Close receiver listening tabs first. This cell records and monitors using one connection.")
    pcm = bytearray()
    underflows = 0
    expected = seconds * 16000 * 2
    try:
        with requests.Session() as session:
            session.trust_env = False
            check = session.get(f"http://{receiver}/", timeout=(5, 5))
            check.raise_for_status()
            with sd.RawOutputStream(samplerate=16000, channels=1, dtype="int16",
                                    device=output_device, latency=0.25) as speakers:
                with session.get(f"http://{receiver}/record.wav", params={"seconds": seconds},
                                 timeout=(5, 15), stream=True) as response:
                    if response.status_code == 404:
                        raise ValueError("Upload the receiver firmware with /record.wav support.")
                    response.raise_for_status()
                    response.raw.decode_content = True

                    def read_exact(length):
                        data = bytearray()
                        while len(data) < length:
                            part = response.raw.read(length - len(data))
                            if not part:
                                raise ValueError("Receiver disconnected before the recording completed.")
                            data.extend(part)
                        return bytes(data)

                    # The receiver endpoint emits a fixed 44-byte PCM WAV header.
                    header = read_exact(44)
                    if (header[:4] != b"RIFF" or header[8:16] != b"WAVEfmt " or
                            header[36:40] != b"data" or
                            struct.unpack_from("<IHHIIHH", header, 16) !=
                            (16, 1, 1, 16000, 32000, 2, 16) or
                            struct.unpack_from("<I", header, 40)[0] != expected):
                        raise ValueError("Unexpected recording header; expected mono 16 kHz, 16-bit WAV.")
                    print(f"Recording and monitoring {seconds} seconds...")
                    # Prebuffer 256 ms to absorb short delivery delays.
                    first = True
                    while len(pcm) < expected:
                        block = read_exact(min(8192 if first else 2048, expected - len(pcm)))
                        first = False
                        pcm.extend(block)  # Preserve original samples in the saved file.
                        if monitor_volume != 1:
                            values = struct.unpack("<" + "h" * (len(block) // 2), block)
                            block = struct.pack("<" + "h" * len(values),
                                *(max(-32768, min(32767, round(v * monitor_volume))) for v in values))
                        underflows += bool(speakers.write(block))
    except KeyboardInterrupt:
        print("Recording interrupted; no complete recording saved.")
        return
    except Exception as exc:
        print(f"Recording/monitoring failed: {exc}")
        print("Check the receiver IP, close its browser stream, and check your speaker device.")
        return
    path = Path.cwd() / f"recording-{datetime.now():%Y%m%d-%H%M%S-%f}.wav"
    with wave.open(str(path), "wb") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(16000)
        wav.writeframes(pcm)
    print(f"Saved: {path}")
    print(f"Speaker buffer underflows: {underflows} (increases can explain monitoring clicks).")
    display(Audio(filename=str(path)))
    return path

recorded_path = record_receiver(receiver, seconds)


Close receiver listening tabs first. This cell records and monitors using one connection.
Recording and monitoring 10 seconds...
Saved: c:\Users\micha\OneDrive - Georgia Southern University\ai security\laser\laser-command-injection\recording-20260905-163259-842948.wav
Speaker buffer underflows: 0 (increases can explain monitoring clicks).


In [19]:
from pathlib import Path
from datetime import datetime
import threading
import queue
import struct
import wave
import requests
import sounddevice as sd
import ipywidgets as widgets
from IPython.display import display

# Stop the previous listener before rerunning this cell.
if "_receiver_ui" in globals() and _receiver_ui["thread"] and _receiver_ui["thread"].is_alive():
    raise RuntimeError("Click Stop listening, wait for 'Stopped', then rerun this cell.")

recordings_folder = Path.cwd() / "recordings"
recordings_folder.mkdir(exist_ok=True)

ip = widgets.Text(value="192.168.1.143", description="Receiver IP:")
start = widgets.Button(description="Start listening", button_style="success")
stop = widgets.Button(description="Stop listening", button_style="warning", disabled=True)
record = widgets.ToggleButton(value=False, description="Recording OFF", icon="circle")
boost = widgets.FloatSlider(value=1, min=0, max=20, step=0.1,
                            description="Volume boost:", readout_format=".1f",
                            continuous_update=True, style={"description_width": "initial"})
status = widgets.Label(value=f"Ready. Files: {recordings_folder}")
log = widgets.Output(layout={"max_height": "180px", "overflow": "auto"})
state = {"thread": None, "stop": threading.Event(), "lock": threading.Lock(),
         "record": False, "gain": 1.0, "pcm": bytearray(), "started": None,
         "number": 0, "speaker_error": None}
_receiver_ui = state
chunk_bytes = 16000 * 2 * 15

def save_locked(partial=False):
    if not state["pcm"]:
        return
    state["number"] += 1
    suffix = "_partial" if partial else ""
    path = recordings_folder / (
        f"{state['started']:%Y%m%d_%H%M%S_%f}_{state['number']:05d}{suffix}.wav")
    with wave.open(str(path), "wb") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(16000)
        wav.writeframes(state["pcm"])
    state["pcm"].clear()
    log.append_stdout(f"Saved {path.name}\n")

def recording_changed(change):
    try:
        with state["lock"]:
            state["record"] = bool(change["new"])
            if not state["record"]:
                save_locked(partial=True)
        record.description = "Recording ON" if change["new"] else "Recording OFF"
        record.button_style = "danger" if change["new"] else ""
    except Exception as exc:
        state["stop"].set()
        log.append_stdout(f"Recording error: {exc}\n")

def gain_changed(change):
    with state["lock"]:
        state["gain"] = float(change["new"])

record.observe(recording_changed, names="value")
boost.observe(gain_changed, names="value")

def listen(address):
    audio_queue = queue.Queue(maxsize=32)
    speaker_done = threading.Event()
    dropped = 0
    underflows = 0

    def play_speakers():
        nonlocal underflows
        try:
            with sd.RawOutputStream(samplerate=16000, channels=1,
                                    dtype="int16", latency=0.25) as speakers:
                while not speaker_done.is_set() and not state["stop"].is_set():
                    try:
                        block = audio_queue.get(timeout=0.1)
                    except queue.Empty:
                        continue
                    with state["lock"]:
                        gain = state["gain"]
                    if gain != 1:
                        values = struct.unpack("<" + "h" * (len(block) // 2), block)
                        block = struct.pack("<" + "h" * len(values),
                            *(max(-32768, min(32767, round(v * gain))) for v in values))
                    underflows += bool(speakers.write(block))
        except Exception as exc:
            state["speaker_error"] = str(exc)
            state["stop"].set()

    player = threading.Thread(target=play_speakers, daemon=True)
    player.start()
    try:
        with requests.Session() as session:
            session.trust_env = False
            with session.get(f"http://{address}/audio", stream=True, timeout=(5, 5)) as response:
                response.raise_for_status()
                if "application/octet-stream" not in response.headers.get("Content-Type", "").lower():
                    raise RuntimeError("Expected receiver 16-bit PCM /audio stream.")
                status.value = "Listening. Toggle Recording ON to save 15-second WAV files."
                pending = b""
                for incoming in response.iter_content(chunk_size=2048):
                    if state["stop"].is_set():
                        break
                    combined = pending + incoming
                    aligned = len(combined) // 2 * 2
                    block, pending = combined[:aligned], combined[aligned:]
                    if not block:
                        continue
                    with state["lock"]:
                        if state["record"]:
                            offset = 0
                            while offset < len(block):
                                if not state["pcm"]:
                                    state["started"] = datetime.now()
                                count = min(chunk_bytes - len(state["pcm"]), len(block) - offset)
                                state["pcm"].extend(block[offset:offset + count])
                                offset += count
                                if len(state["pcm"]) == chunk_bytes:
                                    save_locked()
                    try:
                        audio_queue.put_nowait(block)
                    except queue.Full:
                        dropped += 1
                if not state["stop"].is_set():
                    raise RuntimeError("Receiver disconnected.")
    except Exception as exc:
        if not state["stop"].is_set():
            log.append_stdout(f"Error: {exc}\nClose receiver browser streams and verify the IP.\n")
    finally:
        speaker_done.set()
        player.join(timeout=3)
        if state["speaker_error"]:
            log.append_stdout(f"Speaker error: {state['speaker_error']}\n")
        try:
            with state["lock"]:
                state["record"] = False
                save_locked(partial=True)
        except Exception as exc:
            log.append_stdout(f"Save error: {exc}\n")
        record.value = False
        start.disabled = False
        stop.disabled = True
        ip.disabled = False
        status.value = f"Stopped. Monitor underflows: {underflows}; monitor-only dropped blocks: {dropped}."

def start_clicked(_):
    if state["thread"] and state["thread"].is_alive():
        return
    state["stop"].clear()
    state["speaker_error"] = None
    start.disabled = True
    stop.disabled = False
    ip.disabled = True
    status.value = "Connecting..."
    state["thread"] = threading.Thread(target=listen, args=(ip.value.strip(),), daemon=True)
    state["thread"].start()

def stop_clicked(_):
    state["stop"].set()
    record.value = False
    status.value = "Stopping (up to 5 seconds if the receiver is not responding)..."

start.on_click(start_clicked)
stop.on_click(stop_clicked)
display(widgets.VBox([
    ip, widgets.HBox([start, stop, record]), boost,
    widgets.Label("Boost affects speakers only. Recordings preserve original samples."),
    widgets.Label("Close the receiver browser stream first. Recording OFF saves any partial chunk."),
    status, log
]))


Saving to: c:\Users\micha\OneDrive - Georgia Southern University\ai security\laser\laser-command-injection\receiver_20260905_163312_124449
Close receiver browser streams. Interrupt this cell to stop.
Timestamps use this computer's local time when each chunk begins arriving.
Listening and recording...
Saved 20260905_163312_211075_00001.wav (15.00 seconds)
Saved 20260905_163327_121454_00002.wav (15.00 seconds)
Saved 20260905_163342_119956_00003.wav (15.00 seconds)
Saved 20260905_163357_139629_00004.wav (15.00 seconds)
Saved 20260905_163412_113730_00005.wav (15.00 seconds)
Saved 20260905_163427_089448_00006.wav (15.00 seconds)
Saved 20260905_163442_255815_00007.wav (15.00 seconds)
Saved 20260905_163457_262476_00008.wav (15.00 seconds)
Saved 20260905_163512_306924_00009.wav (15.00 seconds)
Saved 20260905_163527_309455_00010.wav (15.00 seconds)
Saved 20260905_163542_256113_00011.wav (15.00 seconds)
Saved 20260905_163557_329681_00012.wav (15.00 seconds)

Stopped.
Saved 20260905_163612_335055

In [ ]:
from pathlib import Path
from datetime import datetime
import sys
import subprocess
import importlib.util
import queue
import threading
import wave

for package in ("requests", "sounddevice"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import requests
import sounddevice as sd

receiver = "192.168.1.143"
monitor_volume = 1.0       # Speakers only: 0=mute, 1=normal, 2=double
output_device = None       # Default speakers
sample_rate = 16000
chunk_seconds = 15
chunk_bytes = sample_rate * 2 * chunk_seconds

folder = Path.cwd() / datetime.now().strftime("receiver_%Y%m%d_%H%M%S_%f")
folder.mkdir(exist_ok=False)

monitor_queue = queue.Queue(maxsize=32)
stop_event = threading.Event()
monitor_errors = []
monitor_stats = {"dropped": 0, "underflows": 0}
ready = threading.Event()

def monitor():
    import struct

    try:
        with sd.RawOutputStream(
            samplerate=sample_rate, channels=1, dtype="int16",
            device=output_device, latency=0.25
        ) as speakers:
            ready.set()
            while not stop_event.is_set() or not monitor_queue.empty():
                try:
                    block = monitor_queue.get(timeout=0.1)
                except queue.Empty:
                    continue
                if monitor_volume != 1:
                    samples = struct.unpack("<" + "h" * (len(block) // 2), block)
                    block = struct.pack(
                        "<" + "h" * len(samples),
                        *(max(-32768, min(32767, round(v * monitor_volume)))
                          for v in samples)
                    )
                monitor_stats["underflows"] += bool(speakers.write(block))
    except Exception as exc:
        monitor_errors.append(str(exc))
        ready.set()

def save_chunk(data, started, number, partial=False):
    suffix = "_partial" if partial else ""
    name = f"{started:%Y%m%d_%H%M%S_%f}_{number:05d}{suffix}.wav"
    path = folder / name
    with wave.open(str(path), "wb") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(sample_rate)
        wav.writeframes(data)
    print(f"Saved {name} ({len(data) / (sample_rate * 2):.2f} seconds)")

worker = threading.Thread(target=monitor, daemon=True)
recording = bytearray()
pending = b""
started = None
number = 1

print(f"Saving to: {folder}")
print("Close receiver browser streams. Interrupt this cell to stop.")
print("Timestamps use this computer's local time when each chunk begins arriving.")

try:
    if not 0 <= monitor_volume <= 20:
        raise ValueError("monitor_volume must be between 0 and 20.")

    worker.start()
    if not ready.wait(10):
        raise RuntimeError("Speaker initialization timed out.")
    if monitor_errors:
        raise RuntimeError(monitor_errors[0])

    with requests.Session() as session:
        session.trust_env = False
        with session.get(
            f"http://{receiver}/audio",
            stream=True,
            timeout=(5, 15)
        ) as response:
            response.raise_for_status()
            content_type = response.headers.get("Content-Type", "").lower()
            if "application/octet-stream" not in content_type:
                raise RuntimeError(f"Unexpected stream type: {content_type}")

            print("Listening and recording...")
            for incoming in response.iter_content(chunk_size=2048):
                if monitor_errors:
                    raise RuntimeError(f"Speaker error: {monitor_errors[0]}")
                if not incoming:
                    continue

                combined = pending + incoming
                aligned = len(combined) // 2 * 2
                block, pending = combined[:aligned], combined[aligned:]
                if not block:
                    continue

                # Recording never drops samples to accommodate slow speakers.
                offset = 0
                while offset < len(block):
                    if not recording:
                        started = datetime.now()
                    count = min(chunk_bytes - len(recording), len(block) - offset)
                    recording.extend(block[offset:offset + count])
                    offset += count
                    if len(recording) == chunk_bytes:
                        save_chunk(recording, started, number)
                        recording.clear()
                        number += 1

                # Bound monitoring latency without dropping recorded audio.
                try:
                    monitor_queue.put_nowait(block)
                except queue.Full:
                    monitor_stats["dropped"] += 1

            print("Receiver ended the stream.")

except KeyboardInterrupt:
    print("\nStopped.")
except Exception as exc:
    print(f"\nError: {exc}")
finally:
    stop_event.set()
    if worker.is_alive():
        worker.join(timeout=3)
    if recording:
        save_chunk(recording, started, number, partial=True)
    print(f"Folder: {folder}")
    print(
        f"Monitor underflows: {monitor_stats['underflows']}; "
        f"monitor-only dropped blocks: {monitor_stats['dropped']}"
    )

Saving to: c:\Users\micha\OneDrive - Georgia Southern University\ai security\laser\laser-command-injection\receiver_20260905_164458_762094
Close receiver browser streams. Interrupt this cell to stop.
Timestamps use this computer's local time when each chunk begins arriving.
Listening and recording...
Saved 20260905_164458_855824_00001.wav (15.00 seconds)
Saved 20260905_164513_782132_00002.wav (15.00 seconds)
Saved 20260905_164528_756339_00003.wav (15.00 seconds)


In [ ]:
from IPython.display import display
import ipywidgets as widgets

display(widgets.VBox([
    boost,   # Live audio boost slider: 0–20×
    record,  # Recording ON/OFF toggle
]))